# ILP basics

_Combinatorial Optimization course, FEE CTU in Prague. Created by [Industrial Informatics Department](http://industrialinformatics.fel.cvut.cz)._

During the second lab of the course, we concentrate on the (I)LP modeling basics. The goal is to go through several simple examples and try to implement them using the ILP formalism.
In this document, we show the basic syntax on a simple LP problem.
Afterward, several NP-hard problems are formally presented, for which the students should fill the ILP formulations.


## Recap: Linear programming

In the linear programming, we try to solve the following problem:

$$
\min_{x} \{ c^{\text{T}} x \ | \ Ax \leq b \}, \text{ where } c \in \mathbb{R}^{n}, x \in \mathbb{R}^{n}, A \in \mathbb{R}^{m \times n}, b \in \mathbb{R}^{m}
$$

i.e., we want to minimize a linear objective function, subject to linear inequality constraints defininy the feasible region (here, convex polytope).

### Example

Take as an example the following problem:

\begin{align}
         \min_{x,y} {-x + 2y} \ \ \text{s.t.} \\
        -4x -9y \leq -18 \\
        \frac{3}{2}x - y \leq \frac{27}{4} \\
        \frac{8}{17}x - y \geq -2
\end{align}

Rewritten to the matrix form, the same problem would look as follows:

\begin{align}
            \min_{x,y} \begin{bmatrix}
            -1 & 2
            \end{bmatrix} \cdot
            \begin{bmatrix}
            x \\ y
            \end{bmatrix} \ \text{s.t.} \\
            \begin{bmatrix}
            -4 & -9 \\ \frac{3}{2} & -1 \\ -\frac{8}{17} & 1
            \end{bmatrix} \cdot
            \begin{bmatrix}
            x \\ y
            \end{bmatrix} \leq
            \begin{bmatrix}
            -18 \\ \frac{27}{4} \\ 2
            \end{bmatrix}        
\end{align}

For this simple problem, we can visualize the individual constraints, the feasible region and even the optima, which is $x = \frac{9}{2}, y = 0$.

![LP illustration](https://gitlab.com/comb-opt/course/-/raw/main/Google%20Colab%20Resources/lab02/lp.png?ref_type=heads)


Now, let us try to rewrite this problem using the Gurobi syntax. Then, we can call the solver and see if the solution that we have found 'graphically' is, indeed, correct.

The model itself will consist of variables (including definition of their domains) and constraints. Here, the implementation is pretty straightforward:

But before we start, we need to initiate gurobi.

In [7]:
!pip install gurobipy

In [8]:
# Gurobi model for the example problem
import gurobipy as g  # import Gurobi
import numpy as np

# Input data ----------------------
# The program above

# Model ---------------------------
m = g.Model()  # create an empty model

# - variables
x = m.addVar(vtype=g.GRB.CONTINUOUS, name="x")
y = m.addVar(vtype=g.GRB.CONTINUOUS, name="y")

# - constraints
A = np.array([[-4, -9], [3/2, -1], [-8/17, 1]])
B = np.array([-18, 27/4, -2])
C = np.array([-1, 2])
Vars = np.array([x, y])

# m.addConstr(A@Vars >= B)
m.addConstrs(A[i,:]@Vars<=B[i] for i in range(3))

# - objective
# method setObjective is used to set objective
m.setObjective(C@Vars, g.GRB.MINIMIZE)

m.optimize()

# Solution ------------------------
print("x:", x.X)  # use [variable].X to get the optimal value
print("y:", y.X)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Min)
Model fingerprint: 0xb87608af
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-01, 9e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 2e+01]

Presolve removed 3 rows and 2 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -4.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective -4.500000000e+00
x: 4.5
y: 0.0


**While executing the model, Gurobi solver prints some information:**

- `Thread count: 2 physical cores, 4 logical processors, using up to 4 threads`

Some parts of the solver can work in parallel - which might be beneficial when solving complex models.

- `Optimize a model with 3 rows, 2 columns and 6 nonzeros`

Here, we see that the matrix $A$ contains 3 rows (constraints), 2 columns (variables) and 6 non-zero coefficients at total.

```
Presolve removed 3 rows and 2 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -4.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds
```

This parts just says that Gurobi was able to find the solution during the "Presolve" phase. For the interested reader, more about presolving in Gurobi is on webpage here: https://support.gurobi.com/hc/en-us/articles/360024738352-How-does-presolve-work

- `Optimal objective -4.500000000e+00`

The value of the objective function $c^{\text{T}} x $ is -4.5.

## Integer linear programming

Now we solve similar problem, but the domains of (some) variables are restricted to be integers.

$$
\min_{x} \{c^{T} x \ \vert \ Ax \leq b\}, c \in \mathbb{R}^{n}, x \in \mathbb{Z}^{n}, A \in \mathbb{R}^{m \times n}, b \in \mathbb{R}^{m}
$$

- Note that solving the problem as LP and rounding the solution **may not lead to optimal solution** (it might even lead to infeasible solution - see the previous example).   
- Also note that optimizing over a subset will not give better solution (it may only worsen). Therefore, the optimal objective of LP provides a bound for the optimal objective of ILP (lower bound for the minimization).

Examine the following situation and compare the optimal objectives of LP and ILP variants of the same problem:

\begin{align}
\max_{x,y} y \\ \text{ subject to} \\
-x+y \leq 1 \\
2x+3y \leq 12 \\
3x+2y \leq 12 \\
x \geq 0 \\
y \geq 0
\end{align}

![ILP vs LP](https://gitlab.com/comb-opt/course/-/raw/main/Google%20Colab%20Resources/lab02/ILP_LP.png?ref_type=heads)

The area inside of the blue polygon represents the feasible region of the LP formulation, while the red dots represent the feasible solutions of the ILP formulation.

**Now, lets enjoy some fun while implementing the models for some well-known NP-hard problems. The goal is to understand the problems and formalize them in Gurobi language.**

### Problem 1: Two partition


Given multiset $S$ of positive integers, we ask if it can be partitioned to two subsets $S_1$, $S_2$, such that sum of the numbers in $S_1$ equals to the sum of the numbers in $S_2$, and each number is assigned (either to $S_1$ or to $S_2$).

**Example 1** Multiset $S = \{ 1,1,1,2,2,3\}$ can be partitioned, e.g., to $S_1 = \{ 2, 3 \}$,  $S_2 = \{1,1,1,2\}$. Note that it is not a unique solution.

**Example 2** Multiset $S = \{2, 4\}$ cannot be partitioned.



In [3]:
import gurobipy as g

# Example input
S = [100, 50, 50, 50, 20, 20, 10]

# TODO: implement the model and find the solution
# Model ---------------------------
m = g.Model()  # create an empty model

# Vars
x = [0]*7
for i in range(7):
  x[i] = m.addVar(vtype=g.GRB.BINARY, name="x"+str(i))

# Constraints
m.addConstr(g.quicksum(S[i]*x[i] for i in range(7)) == g.quicksum(S[i]*(1-x[i]) for i in range(7)))
# m.addConstr(g.quicksum(S[i]*x[i] for i in range(7)) == g.quicksum(S[i]/2 for i in range(7)))

# - objective
# method setObjective is used to set objective
# m.setObjective(C@Vars, g.GRB.MINIMIZE)

m.optimize()

# Solution ------------------------
for i in range(7):
  print("x"+str(i), x[i].X)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1 rows, 7 columns and 7 nonzeros (Min)
Model fingerprint: 0x3fe8359d
Model has 0 linear objective coefficients
Variable types: 0 continuous, 7 integer (7 binary)
Coefficient statistics:
  Matrix range     [2e+01, 2e+02]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+02, 3e+02]

Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%
x0 1.0
x1 1.0
x2 -0.0
x3 -0.0
x4 -0.0
x5 -0.0
x6 -0.0


### Problem 2: Maximum independent set

Given graph $G = (V,E)$, we are trying to find subset $V^{\star} \subseteq V$, such that $V^{\star}$ is as large as possible and for each pair of vertices $u,v \in V^{\star}$, $u \neq v$, it holds that there is no edge $e = \{u,v\}$ in the original graph.

In [4]:
import gurobipy as g

# Example input
n = 5  # number of vertices
edges = [(0, 1), (2, 3), (0, 4), (3, 1), (3, 4)]  # list of edges

# TODO: implement the model and find the solution
m = g.Model()

# Variables
x = [0]*n
for i in range(n):
  x[i] = m.addVar(vtype=g.GRB.BINARY)

# Constrain
for e in edges:
  m.addConstr(x[e[0]] + x[e[1]] <= 1)

# Objective function
m.setObjective(g.quicksum([x[i] for i in range(n)]), g.GRB.MAXIMIZE)

m.optimize()

for i in range(len(x)):
  print(x[i].X)


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5 rows, 5 columns and 10 nonzeros (Max)
Model fingerprint: 0x11f98127
Model has 5 linear objective coefficients
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 2.0000000
Presolve removed 5 rows and 5 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 2: 3 2 

Optimal solution found (tolerance 1.00e-04)
Best objective 3.000000000000e+00, best bound 3.000000000000e+00, gap 0.0

### Problem 3: Minimum vertex cover

Given graph $G = (V,E)$, we are looking for subset $V^{\star} \subseteq V$, such that $V^{\star}$ is as small as possible and each edge $e = \{u,v\}$ is `covered' by at least one vertex $x \in V^{\star}$, i.e. $\forall e = \{u,v\} \in E \ \exists x \in V^{\star}$ such that $x = u$ or $x = v$.

In [5]:
import gurobipy as g

# Example input
n = 5  # number of vertices
edges = [(0, 1), (2, 3), (0, 4), (3, 1), (3, 4)]  # list of edges

# TODO: implement the model and find the solution
m = g.Model()

# Variables
x = [0]*n
for i in range(n):
  x[i] = m.addVar(vtype=g.GRB.BINARY)

# Constrain
for e in edges:
  m.addConstr(x[e[0]] + x[e[1]] >= 1)

# Objective function
m.setObjective(g.quicksum(x[i] for i in range(n)), g.GRB.MINIMIZE)

m.optimize()

for i in range(len(x)):
  print(x[i].X)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5 rows, 5 columns and 10 nonzeros (Min)
Model fingerprint: 0x8926ef47
Model has 5 linear objective coefficients
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 3.0000000
Presolve removed 5 rows and 5 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 2: 2 3 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.000000000000e+00, best bound 2.000000000000e+00, gap 0.0

### Problem 4: SAT (boolean satisfiability problem)

Given formula $\varphi$ (in CNF), we are looking for truth assignment (True / False) to each variable, such that $\varphi$ is satisfied (evaluates to True).

**Example 1** Formula $\varphi = (x_1 \vee x_2) \wedge (\neg x_2 \vee x_3)$ can be satisfied by $x_1 := \text{True}$, $x_3 := \text{True}$, $x_2 := \text{False}$.

**Example 2** Formula $\varphi = (x_1) \wedge (\neg x_1)$ cannot be satisfied.

In [6]:
import gurobipy as g

# Example input
# SAT formula: (x0 or ~x1) and (~x0 or x1 or x2) <- encode directly into the model

# TODO: implement the model and find the solution
m = g.Model()

x = m.addVars(3, vtype=g.GRB.BINARY)

m.addConstr(x[0] + (1-x[1]) >= 1)
m.addConstr((1-x[0]) + x[1] + x[2] >= 1)

m.optimize()

for i in range(3):
  print(i, x[i].x)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 2 rows, 3 columns and 5 nonzeros (Min)
Model fingerprint: 0xf63e6c55
Model has 0 linear objective coefficients
Variable types: 0 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [0e+00, 0e+00]

Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%
0 -0.0
1 -0.0
2 -0.0
